## Intro figure
In this notebook, we display the intro figure

## Imports

Import standard numerical and plotting libraries such as numpy, pandas, matplotlib, and seaborn.

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns


## Define Parameters

Set up the paths to the stored result tensors for both the static baseline and the dynamic (our proposed) evaluation method.

In [ ]:
static_method_results_path = 'stored_results/static_optimized_evaluation_plot_data.pt'
dynamic_method_results_path = 'stored_results/projected_optimized_plat_evaluation_plot_data.pt'

## Define the functions

Define helper functions to load the result data, process it, and generate matplotlib visualizations comparing the expected censoring times between the static baseline and our dynamic projection method.

In [ ]:

def plot_projected(file_path, high_t=True):
    loaded_data = torch.load(file_path)
    expected_c_all = loaded_data['expected_c'].numpy()
    event_times = loaded_data['event_times'].numpy()
    prior_qs = loaded_data['prior_q'].numpy()
    final_Cs = loaded_data['final_Cs'].numpy()
    time_arrange = np.arange(0, 200)

    if high_t:
        plot_idx = ((final_Cs > event_times) & (abs(prior_qs - event_times) < 20) & (expected_c_all.var(axis=-1) > 0)).nonzero()[0][1].item()
    else:
        plot_idx = ((event_times >= 5) &(event_times <= 10) & (expected_c_all[:, 5] <= expected_c_all[:, 0])& (expected_c_all[:, 8] <= expected_c_all[:, 0]) & (event_times <= final_Cs)).nonzero()[0][0].item()

    stopping_time = min(int(event_times[plot_idx].item()), final_Cs[plot_idx].item())
    plt.plot(time_arrange[:stopping_time+1], expected_c_all[plot_idx][:stopping_time+1], label="Dynamic (ours)",
             linewidth=4, color='tab:green')
    # plt.xlim(-0.5, int(event_times[plot_idx]) + 1)
    return plot_idx, event_times[plot_idx], prior_qs[plot_idx], final_Cs[plot_idx]

def plot_static(file_path, plot_idx):
    loaded_data = torch.load(file_path)
    expected_c_all = loaded_data['expected_c'].numpy()
    expected_c_all = expected_c_all[:, None].repeat(repeats=200, axis=-1)
    event_times = loaded_data['event_times'].numpy()
    prior_qs = loaded_data['prior_qs'].numpy()
    final_Cs = loaded_data['final_Cs'].numpy()
    time_arrange = np.arange(0, 200)
    stopping_time = min(int(event_times[plot_idx].item()), final_Cs[plot_idx].item())
    if stopping_time == 0:
        stopping_time = int(event_times[plot_idx].item())
    plt.plot(time_arrange[:stopping_time+1], expected_c_all[plot_idx][:stopping_time+1], label="Static (baseline)",
             linewidth=4, color='tab:blue')

    return plot_idx, event_times[plot_idx], prior_qs[plot_idx], final_Cs[plot_idx]



def plot_censoring_times(high_t):
    assert (torch.load(static_method_results_path)['event_times'].sort().values ==
            torch.load(dynamic_method_results_path)['event_times'].sort().values).all()
    plt.figure(figsize=(7, 5.5))
    plt.rcParams.update({'font.size': 24})
    plot_idx, event_time, prior_q, final_c = plot_projected(
        'stored_results/projected_optimized_plat_evaluation_plot_data.pt', high_t)
    plot_idx, event_time2, prior_q2, final_c2 = plot_static('stored_results/static_optimized_evaluation_plot_data.pt', int(plot_idx))
    print(f"static optimized final_c: {final_c2}")
    assert prior_q == prior_q2 and event_time == event_time2

    plt.axvline(x=event_time, color='blue', linestyle='--', linewidth=4, label=f'True Event Time ({int(event_time)})')

    # plt.title(f"")
    plt.xlabel(r"Conversation Turn $t$")
    plt.ylabel("Expected Censoring Time")
    plt.legend()
    if high_t:
        plt.savefig("figures/expected_censoring_times_high_t.png", bbox_inches='tight', dpi=300)
    else:
        plt.savefig("figures/expected_censoring_times_low_t.png", bbox_inches='tight', dpi=300)
    plt.show()


## Plot the Figure

Execute the plotting functions to generate and save the figures for both high and low target times.

In [ ]:
plot_censoring_times(high_t=True)
plot_censoring_times(high_t=False)